In [0]:
CATALOG = "my_living_index"

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Load Gold tables ──
report_pd = spark.table(f"{CATALOG}.gold.gold_cost_of_living_report").toPandas()
afford_pd = spark.table(f"{CATALOG}.gold.gold_affordability_by_state").toPandas()
inequality_pd = spark.table(f"{CATALOG}.gold.gold_income_inequality").toPandas()

report_pd = report_pd.sort_values("year")
inequality_pd = inequality_pd.sort_values(["year", "income_group"])


# ════════════════════════════════════════════
# CHART 1 — Nominal vs Real Income Over Time
# Matplotlib: clean summary version
# ════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(report_pd["year"], report_pd["income_mean"],
        marker="o", linewidth=2, color="#1f77b4", label="Nominal Income (Mean)")
ax.plot(report_pd["year"], report_pd["income_median"],
        marker="o", linewidth=2, color="#aec7e8", linestyle="--", label="Nominal Income (Median)")
ax.plot(report_pd["year"], report_pd["real_income_mean"],
        marker="s", linewidth=2, color="#ff7f0e", label="Real Income Mean (2010 RM)")
ax.plot(report_pd["year"], report_pd["real_income_median"],
        marker="s", linewidth=2, color="#ffbb78", linestyle="--", label="Real Income Median (2010 RM)")

# Shade the gap between nominal and real
ax.fill_between(report_pd["year"],
                report_pd["income_mean"],
                report_pd["real_income_mean"],
                alpha=0.08, color="#1f77b4", label="Inflation gap")

ax.set_title("Malaysia: Nominal vs Real Household Income (2002–2022)\nBase Year: 2010",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Monthly Household Income (RM)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"RM {x:,.0f}"))
ax.set_xticks(report_pd["year"])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Plotly: interactive version ──
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=report_pd["year"], y=report_pd["income_mean"],
    mode="lines+markers", name="Nominal Income (Mean)",
    line=dict(color="#1f77b4", width=2),
    hovertemplate="Year: %{x}<br>Nominal Mean: RM %{y:,.0f}<extra></extra>"
))
fig1.add_trace(go.Scatter(
    x=report_pd["year"], y=report_pd["income_median"],
    mode="lines+markers", name="Nominal Income (Median)",
    line=dict(color="#aec7e8", width=2, dash="dash"),
    hovertemplate="Year: %{x}<br>Nominal Median: RM %{y:,.0f}<extra></extra>"
))
fig1.add_trace(go.Scatter(
    x=report_pd["year"], y=report_pd["real_income_mean"],
    mode="lines+markers", name="Real Income Mean (2010 RM)",
    line=dict(color="#ff7f0e", width=2),
    hovertemplate="Year: %{x}<br>Real Mean: RM %{y:,.0f}<extra></extra>"
))
fig1.add_trace(go.Scatter(
    x=report_pd["year"], y=report_pd["real_income_median"],
    mode="lines+markers", name="Real Income Median (2010 RM)",
    line=dict(color="#ffbb78", width=2, dash="dash"),
    hovertemplate="Year: %{x}<br>Real Median: RM %{y:,.0f}<extra></extra>"
))

fig1.update_layout(
    title="Malaysia: Nominal vs Real Household Income (2002–2022)",
    xaxis_title="Year",
    yaxis_title="Monthly Household Income (RM)",
    yaxis_tickprefix="RM ",
    yaxis_tickformat=",",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
    height=500
)
fig1.show()


# ════════════════════════════════════════════
# CHART 2 — Income Growth vs Inflation (Bar)
# ════════════════════════════════════════════
chart2 = report_pd.dropna(subset=["mean_growth_pct", "yoy_inflation_pct"]).copy()

# Matplotlib
fig, ax = plt.subplots(figsize=(13, 5))

bar_width = 0.35
x = range(len(chart2))

ax.bar([i - bar_width/2 for i in x], chart2["mean_growth_pct"],
       width=bar_width, label="Nominal Income Growth %",
       color="#1f77b4", alpha=0.85)
ax.bar([i + bar_width/2 for i in x], chart2["yoy_inflation_pct"],
       width=bar_width, label="CPI Inflation %",
       color="#d62728", alpha=0.85)

# Colour x-axis labels red where inflation beat income
for i, (_, row) in enumerate(chart2.iterrows()):
    color = "red" if not row["income_beats_inflation"] else "black"
    ax.get_xticklabels()  # ensure ticks exist
    
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(chart2["year"].astype(str), rotation=45)
ax.set_title("Malaysia: Income Growth vs CPI Inflation per Survey Period\n(Red x-label = inflation won)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Change (%)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Plotly
fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=chart2["year"].astype(str),
    y=chart2["mean_growth_pct"],
    name="Nominal Income Growth %",
    marker_color="#1f77b4",
    hovertemplate="Year: %{x}<br>Income Growth: %{y:.1f}%<extra></extra>"
))
fig2.add_trace(go.Bar(
    x=chart2["year"].astype(str),
    y=chart2["yoy_inflation_pct"],
    name="CPI Inflation %",
    marker_color="#d62728",
    hovertemplate="Year: %{x}<br>Inflation: %{y:.1f}%<extra></extra>"
))
fig2.add_trace(go.Scatter(
    x=chart2["year"].astype(str),
    y=chart2["real_growth_gap"],
    name="Real Growth Gap (Income - Inflation)",
    mode="lines+markers",
    line=dict(color="#2ca02c", width=2, dash="dot"),
    hovertemplate="Year: %{x}<br>Gap: %{y:.1f}%<extra></extra>"
))

fig2.add_hline(y=0, line_color="black", line_width=1)
fig2.update_layout(
    title="Malaysia: Income Growth vs Inflation per Survey Period",
    xaxis_title="Survey Year",
    yaxis_title="Change (%)",
    barmode="group",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
    height=500
)
fig2.show()


# ════════════════════════════════════════════
# CHART 3 — Affordability by State (2022)
# ════════════════════════════════════════════

# Matplotlib
fig, ax = plt.subplots(figsize=(13, 6))

afford_sorted = afford_pd.sort_values("expenditure_to_income_ratio", ascending=True)

colors = afford_sorted["affordability_tier"].map({
    "Comfortable": "#2ca02c",
    "Moderate":    "#ff7f0e",
    "Stretched":   "#d62728",
    "Critical":    "#7f0000"
})

bars = ax.barh(afford_sorted["state"],
               afford_sorted["expenditure_to_income_ratio"],
               color=colors, alpha=0.85)

ax.axvline(x=1.0, color="red", linestyle="--", linewidth=1.2, label="Spending = Income (ratio 1.0)")
ax.axvline(x=afford_sorted["expenditure_to_income_ratio"].mean(),
           color="grey", linestyle=":", linewidth=1.2,
           label=f"National avg ({afford_sorted['expenditure_to_income_ratio'].mean():.2f})")

ax.set_title("Malaysia: Expenditure-to-Income Ratio by State (2022)\nLower = More Affordable",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Expenditure / Income Ratio")

# Add RM surplus annotation
for bar, (_, row) in zip(bars, afford_sorted.iterrows()):
    ax.text(bar.get_width() + 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"RM {row['monthly_surplus']:,.0f} surplus",
            va="center", fontsize=8, color="gray")

ax.legend()
plt.tight_layout()
plt.show()

# Plotly
fig3 = px.bar(
    afford_sorted,
    x="expenditure_to_income_ratio",
    y="state",
    orientation="h",
    color="affordability_tier",
    color_discrete_map={
        "Comfortable": "#2ca02c",
        "Moderate":    "#ff7f0e",
        "Stretched":   "#d62728",
        "Critical":    "#7f0000"
    },
    custom_data=["income_mean", "expenditure_mean", "monthly_surplus", "poverty"],
    title="Malaysia: Affordability by State (2022)"
)

fig3.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Ratio: %{x:.3f}<br>"
        "Mean Income: RM %{customdata[0]:,.0f}<br>"
        "Mean Expenditure: RM %{customdata[1]:,.0f}<br>"
        "Monthly Surplus: RM %{customdata[2]:,.0f}<br>"
        "Poverty Rate: %{customdata[3]:.1f}%"
        "<extra></extra>"
    )
)
fig3.add_vline(x=1.0, line_dash="dash", line_color="red",
               annotation_text="Spending = Income")
fig3.update_layout(
    xaxis_title="Expenditure / Income Ratio",
    yaxis_title="State",
    legend_title="Affordability Tier",
    height=600
)
fig3.show()


# ════════════════════════════════════════════
# CHART 4 — B40 / M40 / T20 Distribution (2024)
# ════════════════════════════════════════════
pct_pd = spark.table(f"{CATALOG}.silver.silver_income_percentile").toPandas()
pct_pd = pct_pd.sort_values("percentile")

# Matplotlib
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: income curve by percentile
color_map = {"B40": "#d62728", "M40": "#ff7f0e", "T20": "#1f77b4"}
for group, grp_df in pct_pd.groupby("income_group"):
    axes[0].plot(grp_df["percentile"], grp_df["income_mean"],
                 color=color_map[group], linewidth=2, label=group)
    axes[0].fill_between(grp_df["percentile"], grp_df["income_minimum"],
                         grp_df["income_maximum"],
                         alpha=0.1, color=color_map[group])

axes[0].set_title("Income Distribution by Percentile (2024)", fontweight="bold")
axes[0].set_xlabel("Percentile")
axes[0].set_ylabel("Monthly Income (RM)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"RM {x:,.0f}"))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: avg income by group bar chart
grp_summary = inequality_pd.copy()
bar_colors = [color_map.get(g, "gray") for g in grp_summary["income_group"]]
axes[1].bar(grp_summary["income_group"], grp_summary["avg_income"],
            color=bar_colors, alpha=0.85)

for i, (_, row) in enumerate(grp_summary.iterrows()):
    axes[1].text(i, row["avg_income"] + 100,
                 f"RM {row['avg_income']:,.0f}",
                 ha="center", fontsize=10, fontweight="bold")
    if pd.notna(row.get("t20_to_b40_ratio")):
        axes[1].set_xlabel(f"T20 earns {row['t20_to_b40_ratio']:.1f}x more than B40", fontsize=10)

axes[1].set_title("Average Income by Group (2024)", fontweight="bold")
axes[1].set_ylabel("Average Monthly Income (RM)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"RM {x:,.0f}"))
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("B40 / M40 / T20 Income Distribution — Malaysia 2024",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Plotly
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Income Curve by Percentile", "Average Income by Group"]
)

for group, grp_df in pct_pd.groupby("income_group"):
    color = color_map[group]
    fig4.add_trace(go.Scatter(
        x=grp_df["percentile"], y=grp_df["income_mean"],
        mode="lines", name=group,
        line=dict(color=color, width=2),
        hovertemplate="Percentile: %{x}<br>Mean Income: RM %{y:,.0f}<extra></extra>"
    ), row=1, col=1)

fig4.add_trace(go.Bar(
    x=grp_summary["income_group"],
    y=grp_summary["avg_income"],
    marker_color=[color_map.get(g, "gray") for g in grp_summary["income_group"]],
    showlegend=False,
    hovertemplate="Group: %{x}<br>Avg Income: RM %{y:,.0f}<extra></extra>"
), row=1, col=2)

fig4.update_layout(
    title_text="B40 / M40 / T20 Income Distribution — Malaysia 2024",
    height=500,
    hovermode="x"
)
fig4.update_yaxes(tickprefix="RM ", tickformat=",")
fig4.show()


# ════════════════════════════════════════════
# FINAL VERDICT — Text Summary
# ════════════════════════════════════════════
latest       = report_pd.dropna(subset=["real_income_growth_pct"]).iloc[-1]
years_losing = report_pd[report_pd["income_beats_inflation"] == False]["year"].tolist()
years_total  = report_pd.dropna(subset=["income_beats_inflation"]).shape[0]
years_won    = report_pd[report_pd["income_beats_inflation"] == True].shape[0]

t20_b40 = inequality_pd["t20_to_b40_ratio"].dropna().iloc[0] \
          if "t20_to_b40_ratio" in inequality_pd.columns else "N/A"

print("=" * 65)
print("  ARE MALAYSIANS ACTUALLY GETTING POORER?")
print("=" * 65)
print(f"\n  Survey years analysed     : {int(report_pd['year'].min())} – {int(report_pd['year'].max())}")
print(f"  Income beat inflation     : {years_won}/{years_total} survey periods")
print(f"  Years purchasing power    ")
print(f"  declined                  : {years_losing}")
print(f"\n  Latest period ({int(latest['year'])})")
print(f"  Nominal mean income       : RM {latest['income_mean']:,.0f}/month")
print(f"  Real mean income (2010 RM): RM {latest['real_income_mean']:,.0f}/month")
print(f"  Nominal income growth     : {latest['mean_growth_pct']:.1f}%")
print(f"  CPI inflation             : {latest['yoy_inflation_pct']:.1f}%")
print(f"  Real income growth        : {latest['real_income_growth_pct']:.1f}%")
print(f"  Status                    : {latest['purchasing_power_status']}")
print(f"\n  Inequality (2024)")
print(f"  T20 earns                 : {t20_b40}x more than B40")
print("=" * 65)
print("\n  VERDICT: In most survey periods, Malaysians saw real income")
print("  grow faster than inflation. However 2020 was a clear setback.")
print("  The bigger concern is inequality — the T20/B40 gap remains wide.")
print("=" * 65)